# 02 – Silver kuralları keşfi

## Hücre 1 – Spark session ve bronze verisi

**Amaç:** Spark session'ı açmak ve bronze trip tablosunu okumak.

**Çıktının okunuşu:**
- `Bronze rows` → 9,554,778 olmalı (bronze job'undaki kontrolle aynı sayı).
- Schema'da 21 kolon olmalı: kaynaktan gelen 19 kolon + `_ingested_at` ve `_source_file`.
- `tpep_pickup_datetime` ve `tpep_dropoff_datetime` tipi `timestamp_ntz` olmalı.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Notebook'ta JVM getOrCreate() anında başladığı için driver memory ayarı burada etkili olur
# (spark-submit'te etkisizdi).
# Session timezone UTC: süre hesabında unix_timestamp, timestamp_ntz değerlerini bu timezone'da yorumlar;
# UTC'de yaz saati geçişi olmadığı için süreler kaymaz.
spark = (
    SparkSession.builder
    .appName("silver-rules-exploration")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

BRONZE_TRIPS_PATH = "/home/jovyan/work/data/bronze/yellow_trips"

df_bronze = spark.read.parquet(BRONZE_TRIPS_PATH)

row_count = df_bronze.count()
print(f"Bronze rows: {row_count:,}")

df_bronze.printSchema()

Bronze rows: 9,554,778
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)



## Hücre 2 – Keşif için yardımcı kolonlar

**Amaç:** Kuralları ölçebilmek için gereken kolonları eklemek:
- `trip_distance_km`: mesafe km olarak (kaynaktaki `trip_distance` mil cinsinden)
- `trip_duration_min`: yolculuk süresi (dakika)
- `avg_speed_kmh`: ortalama hız (km/h)
- `pickup_year`, `pickup_month`: pickup zamanının yılı ve ayı
- `file_year`, `file_month`: satırın geldiği dosyanın yılı ve ayı (dosya adından)

**Çıktının okunuşu:**
- İlk 5 satırda değerler mantıklı olmalı: birkaç km, 5-30 dakika, 10-40 km/h civarı.
- `pickup_year/month` ile `file_year/month` normalde aynı olmalı. Farklı olanları Hücre 4'te sayacağız.

In [2]:
MILES_TO_KM = 1.609344

# Dosya adı örneği: yellow_tripdata_2024-01.parquet
# (\d{4}) = 1. grup, yıl; (\d{2}) = 2. grup, ay
SOURCE_FILE_PATTERN = r"_(\d{4})-(\d{2})\.parquet"

# Mesafe
distance_km = F.col("trip_distance") * MILES_TO_KM
df_explore = df_bronze.withColumn("trip_distance_km", distance_km)

# Süre: unix_timestamp zamanı saniyeye çevirir; iki saniyenin farkı / 60 = dakika
pickup_seconds = F.unix_timestamp("tpep_pickup_datetime")
dropoff_seconds = F.unix_timestamp("tpep_dropoff_datetime")
duration_seconds = dropoff_seconds - pickup_seconds
duration_min = duration_seconds / 60
df_explore = df_explore.withColumn("trip_duration_min", duration_min)

# Hız: süre 0 ise normal bölme ANSI mode'da hata verir; try_divide hata yerine null döner
duration_hours = F.col("trip_duration_min") / 60
speed_kmh = F.try_divide(F.col("trip_distance_km"), duration_hours)
df_explore = df_explore.withColumn("avg_speed_kmh", speed_kmh)

# Pickup yılı ve ayı
pickup_year = F.year("tpep_pickup_datetime")
pickup_month = F.month("tpep_pickup_datetime")
df_explore = df_explore.withColumn("pickup_year", pickup_year)
df_explore = df_explore.withColumn("pickup_month", pickup_month)

# Dosya yılı ve ayı: regexp_extract string döner ("01"), karşılaştırma için int'e çeviriyoruz
file_year_text = F.regexp_extract("_source_file", SOURCE_FILE_PATTERN, 1)
file_month_text = F.regexp_extract("_source_file", SOURCE_FILE_PATTERN, 2)
df_explore = df_explore.withColumn("file_year", file_year_text.cast("int"))
df_explore = df_explore.withColumn("file_month", file_month_text.cast("int"))

preview_columns = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_distance_km",
    "trip_duration_min",
    "avg_speed_kmh",
    "pickup_year",
    "pickup_month",
    "file_year",
    "file_month",
]
df_explore.select(preview_columns).show(5, truncate=False)

+--------------------+---------------------+------------------+------------------+------------------+-----------+------------+---------+----------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance_km  |trip_duration_min |avg_speed_kmh     |pickup_year|pickup_month|file_year|file_month|
+--------------------+---------------------+------------------+------------------+------------------+-----------+------------+---------+----------+
|2024-01-01 00:57:55 |2024-01-01 01:17:43  |2.7680716800000003|19.8              |8.388096          |2024       |1           |2024     |1         |
|2024-01-01 00:03:00 |2024-01-01 00:09:36  |2.8968192000000004|6.6               |26.334720000000004|2024       |1           |2024     |1         |
|2024-01-01 00:17:06 |2024-01-01 00:35:01  |7.563916800000001 |17.916666666666668|25.33032602790698 |2024       |1           |2024     |1         |
|2024-01-01 00:36:38 |2024-01-01 00:44:56  |2.2530816         |8.3               |16.28733686746988 |2024       

## Hücre 3 – Sayısal kolonların dağılımı

**Amaç:** Threshold seçmek için mesafe, süre, hız, ücret ve yolcu sayısının dağılımını görmek.

**Çıktının okunuşu:**
- Her satır bir istatistik, her kolon bir metrik.
- `count`: null olmayan değer sayısı. `avg_speed_kmh`'te daha az çıkarsa fark, süresi 0 olan (hızı null kalan) satırlardır.
- `min`, `0.1%`, `1%`: alt kuyruk. Sıfır ve negatif değerler burada görünür.
- `50%` (median): tipik yolculuk.
- `99%`, `99.9%`, `max`: üst kuyruk. 99.9% ile max arasında büyük bir sıçrama varsa kuyruktaki değerler büyük ihtimalle hatalı kayıttır.
- Değerler string döner, percentile'lar yaklaşık hesaplanır. Okumak için yeterli.

In [3]:
metric_columns = [
    "trip_distance_km",
    "trip_duration_min",
    "avg_speed_kmh",
    "fare_amount",
    "total_amount",
    "passenger_count",
]

# Ortalama outlier'lardan çok etkilenir; dağılımı percentile'larla görüyoruz
statistics = ["count", "min", "0.1%", "1%", "50%", "99%", "99.9%", "max"]

df_metrics = df_explore.select(metric_columns)
df_summary = df_metrics.summary(*statistics)

df_summary.show(truncate=False)

+-------+-----------------+-------------------+-------------------+-----------+------------+---------------+
|summary|trip_distance_km |trip_duration_min  |avg_speed_kmh      |fare_amount|total_amount|passenger_count|
+-------+-----------------+-------------------+-------------------+-----------+------------+---------------+
|count  |9554778          |9554778            |9552157            |9554778    |9554778     |8802816        |
|min    |0.0              |-52.06666666666667 |-45537.997824000005|-999.0     |-1000.0     |0              |
|0.1%   |0.0              |0.06666666666666667|0.0                |-70.0      |-74.0       |0              |
|1%     |0.0              |0.6833333333333333 |0.0                |-5.1       |-9.8        |0              |
|50%    |2.7358848        |12.05              |15.240922637837837 |12.8       |20.2        |1              |
|99%    |31.96157184      |63.71666666666667  |57.51038117647059  |75.0       |103.34      |5              |
|99.9%  |46.9767513

## Hücre 4 – Aday kurallar kaç satırı etkiliyor?

**Amaç:** Her aday cleaning kuralının kaç satırı yakaladığını ve tüm kurallar uygulansa toplam kaç satırın atılacağını görmek.
Threshold'lar: süre 180 dakika, mesafe 150 km, hız 120 km/h.

**Çıktının okunuşu:**
- Her kural için: yakaladığı satır sayısı ve toplama oranı (%).
- Kurallar birbirinden bağımsız sayılır. Bir satır birden fazla kuralı ihlal edebilir, bu yüzden kural sayılarının toplamı `any_rule`'dan büyük olabilir.
- `any_rule`: en az bir kuralı ihlal eden satır sayısı = tüm kurallar uygulansaydı atılacak satır sayısı.
- Oranı çok yüksek çıkan bir kural varsa threshold'u ya da kuralın kendisi yeniden düşünülür.

In [50]:
MAX_DURATION_MIN = 180
MAX_DISTANCE_KM = 150
MAX_SPEED_KMH = 120

zero_or_negative_distance = F.col("trip_distance_km") <= 0
zero_or_negative_fare_amount = F.col("fare_amount") <= 0
zero_or_negative_total_amount = F.col("total_amount") <= 0
dropoff_before_pickup = F.col("trip_duration_min") < 0
zero_duration = F.col("trip_duration_min") == 0

pickup_year_differs = F.col("pickup_year") != F.col("file_year")
pickup_month_differs = F.col("pickup_month") != F.col("file_month")
pickup_outside_file_month = pickup_year_differs | pickup_month_differs

zero_passengers = F.col("passenger_count") == 0
too_long_duration = F.col("trip_duration_min") > MAX_DURATION_MIN
too_long_distance = F.col("trip_distance_km") > MAX_DISTANCE_KM
too_fast = F.col("avg_speed_kmh") > MAX_SPEED_KMH

# En az bir kuralı ihlal eden satırlar
any_rule = (
    zero_or_negative_distance
    | zero_or_negative_fare_amount
    | zero_or_negative_total_amount
    | dropoff_before_pickup
    | zero_duration
    | pickup_outside_file_month
    | zero_passengers
    | too_long_duration
    | too_long_distance
    | too_fast
)

# Tüm sayımlar tek select'te: veri bir kez okunur.
# True → 1, False → 0; sum null'ları saymaz.
df_counts = df_explore.select(
    F.count("*").alias("total_rows"),
    F.sum(zero_or_negative_distance.cast("int")).alias("zero_or_negative_distance"),
    F.sum(zero_or_negative_fare_amount.cast("int")).alias("zero_or_negative_fare_amount"),
    F.sum(zero_or_negative_total_amount.cast("int")).alias("zero_or_negative_total_amount"),
    F.sum(dropoff_before_pickup.cast("int")).alias("dropoff_before_pickup"),
    F.sum(zero_duration.cast("int")).alias("zero_duration"),
    F.sum(pickup_outside_file_month.cast("int")).alias("pickup_outside_file_month"),
    F.sum(zero_passengers.cast("int")).alias("zero_passengers"),
    F.sum(too_long_duration.cast("int")).alias("too_long_duration"),
    F.sum(too_long_distance.cast("int")).alias("too_long_distance"),
    F.sum(too_fast.cast("int")).alias("too_fast"),
    F.sum(any_rule.cast("int")).alias("any_rule"),
)

#df_counts.show(vertical=True)

row_counts = df_counts.first()
total_rows = row_counts["total_rows"]

for column_name in df_counts.columns:
    rule_count = row_counts[column_name]
    rule_pct = rule_count / total_rows * 100
    print(f"{column_name}: {rule_count:,} ({rule_pct:.2f}%)")

total_rows: 9,554,778 (100.00%)
zero_or_negative_distance: 215,764 (2.26%)
zero_or_negative_fare_amount: 139,596 (1.46%)
zero_or_negative_total_amount: 117,187 (1.23%)
dropoff_before_pickup: 180 (0.00%)
zero_duration: 2,621 (0.03%)
pickup_outside_file_month: 56 (0.00%)
zero_passengers: 105,931 (1.11%)
too_long_duration: 6,103 (0.06%)
too_long_distance: 271 (0.00%)
too_fast: 3,377 (0.04%)
any_rule: 455,137 (4.76%)


## Hücre 5 – Duplicate kayıtlar

**Amaç:** Veride birebir tekrar eden yolculuk var mı, kaç tane?
Veride trip ID yok. Duplicate tanımı: kaynaktan gelen tüm kolonları aynı olan satırlar.

**Çıktının okunuşu:**
- `business columns`: karşılaştırmaya giren kolon sayısı. 19 olmalı (21 kolon − 2 metadata kolonu).
- `unique rows`: tekilleştirme sonrası satır sayısı.
- `duplicate rows` = toplam − tekil. 0 ise silver'da duplicate kuralına gerek yok.

In [52]:
# Metadata kolonları (_ingested_at, _source_file) karşılaştırmaya girmez:
# aynı yolculuk iki farklı dosyada olsa _source_file farklı olur ve duplicate yakalanmaz
business_columns = []
for column_name in df_bronze.columns:
    if not column_name.startswith("_"):
        business_columns.append(column_name)

print(f"business columns: {len(business_columns)}")

# dropDuplicates satırları karşılaştırmak için shuffle yapar
# (veriyi kolon değerlerine göre çekirdekler arasında yeniden dağıtır); bu hücrenin en pahalı kısmı
df_unique = df_bronze.dropDuplicates(business_columns)
unique_rows = df_unique.count()

duplicate_rows = total_rows - unique_rows

print(f"total rows: {total_rows:,}")
print(f"unique rows: {unique_rows:,}")
print(f"duplicate rows: {duplicate_rows:,}")

business columns: 19
total rows: 9,554,778
unique rows: 9,554,777
duplicate rows: 1


## Hücre 6 – Sıfır/negatif tutarlar ve sıfır mesafe hangi ödeme tipinden?

**Amaç:** Hücre 4'teki en büyük iki grubun (sıfır/negatif tutar, sıfır mesafe) gerçek yolculuk mu yoksa iptal/iade kaydı mı olduğunu anlamak.

`payment_type` kodları (TLC, Mart 2025): 0 = Flex Fare, 1 = Credit card, 2 = Cash, 3 = No charge, 4 = Dispute, 5 = Unknown, 6 = Voided trip.

**Çıktının okunuşu:**
- İlk tablo: `total_amount <= 0` satırlarının ödeme tipine göre dağılımı. Çoğu 3, 4 veya 6 ise bunlar iptal, itiraz veya iade kayıtlarıdır ve atılmaları doğrudur.
- İkinci tablo: `trip_distance_km = 0` satırlarının dağılımı. Çoğu 1 veya 2 ise ücreti ödenmiş ama mesafesi kaydedilmemiş yolculuklar olabilir. O zaman kuralı yeniden konuşuruz.

In [56]:
df_zero_total_by_payment = df_explore.filter(F.col("total_amount") <= 0).groupBy("payment_type").count().orderBy("payment_type")
df_zero_distance_by_payment = df_explore.filter(F.col("trip_distance_km") == 0).groupBy("payment_type").count().orderBy("payment_type")

print("total_amount <= 0")
df_zero_total_by_payment.show()

print("trip_distance_km = 0")
df_zero_distance_by_payment.show()

total_amount <= 0
+------------+-----+
|payment_type|count|
+------------+-----+
|           0|   73|
|           1|  411|
|           2|27735|
|           3|18509|
|           4|70459|
+------------+-----+

trip_distance_km = 0
+------------+------+
|payment_type| count|
+------------+------+
|           0|101379|
|           1| 61527|
|           2| 27009|
|           3| 13558|
|           4| 12291|
+------------+------+

